# AI Programming — Lecture 9
## Stabilizing Neural Network Training

이 노트북에서는 Lecture 9의 핵심 내용을 하나의 흐름으로 실습합니다.

1. **Activation Functions**
2. **Input Scaling**
3. **Batch Normalization**
4. **Layer Normalization**

이번 강의의 핵심 질문은 다음과 같습니다.

> **신경망의 activation과 gradient를 layer 전체에서 안정적으로 유지하려면 어떻게 해야 할까?**

### 학습 목표

실습을 마치면 다음 내용을 설명할 수 있어야 합니다.

- Sigmoid, tanh, ReLU, Leaky ReLU의 차이를 설명할 수 있습니다.
- Sigmoid에서 vanishing gradient가 발생할 수 있는 이유를 확인할 수 있습니다.
- ReLU의 dying ReLU 문제와 Leaky ReLU의 차이를 이해합니다.
- Normalization과 standardization의 차이를 구분할 수 있습니다.
- Input scaling은 training data의 통계량으로 `fit`해야 한다는 점을 이해합니다.
- Batch Normalization이 mini-batch 방향으로 각 neuron을 정규화한다는 점을 확인할 수 있습니다.
- Batch Normalization의 learnable scale/shift parameter인 $\gamma$, $\beta$의 역할을 이해합니다.
- Layer Normalization이 sample별 feature 방향으로 정규화한다는 점을 설명할 수 있습니다.
- BatchNorm과 LayerNorm의 normalization axis 차이를 코드로 확인할 수 있습니다.

### 실습 방법

1. 셀을 위에서부터 순서대로 실행하세요.
2. 그래프와 출력에서 **평균(mean), 표준편차(std), gradient의 크기**를 확인하세요.
3. `TODO`가 표시된 값은 직접 변경해 다시 실행하세요.
4. 이번 실습에서는 최종 accuracy보다 **activation과 gradient가 어떻게 변하는지**를 관찰하는 것이 더 중요합니다.

## 0. 라이브러리 불러오기

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=5, suppress=True)

# Part I. Activation Functions

## 1. Activation Functions 비교

Lecture 9에서는 다음 네 activation function을 비교합니다.

### Sigmoid

$$
\sigma(x)=\frac{1}{1+e^{-x}}
$$

### Tanh

$$
\tanh(x)=\frac{e^x-e^{-x}}{e^x+e^{-x}}
$$

### ReLU

$$
\mathrm{ReLU}(x)=\max(0,x)
$$

### Leaky ReLU

$$
\mathrm{LeakyReLU}(x)=\max(\alpha x, x)
$$

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def tanh(x):
    return np.tanh(x)

def relu(x):
    return np.maximum(0.0, x)

def leaky_relu(x, alpha=0.1):
    return np.where(x >= 0, x, alpha * x)

x = np.linspace(-10, 10, 500)

plt.plot(x, sigmoid(x), label="Sigmoid")
plt.plot(x, tanh(x), label="tanh")
plt.plot(x, relu(x), label="ReLU")
plt.plot(x, leaky_relu(x), label="Leaky ReLU")
plt.axhline(0, linewidth=0.8)
plt.axvline(0, linewidth=0.8)
plt.xlabel("x")
plt.ylabel("activation")
plt.title("Activation Functions")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 확인할 내용

- Sigmoid 출력은 `(0, 1)` 범위입니다.
- tanh 출력은 `(-1, 1)` 범위이며 zero-centered입니다.
- ReLU는 음수 영역을 0으로 만듭니다.
- Leaky ReLU는 음수 영역에서도 작은 기울기를 유지합니다.

## 2. Sigmoid와 Vanishing Gradient

Sigmoid의 derivative는

$$
\frac{d\sigma(x)}{dx}
=
\sigma(x)(1-\sigma(x))
$$

입니다.

Sigmoid derivative의 최댓값은 0.25이며,
입력의 절댓값이 커지면 derivative는 0에 가까워집니다.

In [ ]:
def sigmoid_grad(x):
    s = sigmoid(x)
    return s * (1.0 - s)

plt.plot(x, sigmoid_grad(x))
plt.axhline(0.25, linestyle="--")
plt.xlabel("x")
plt.ylabel("d sigmoid / dx")
plt.title("Sigmoid Derivative")
plt.grid(alpha=0.3)
plt.show()

print("Maximum derivative:", sigmoid_grad(x).max())

### Vanishing Gradient의 간단한 직관

깊은 network의 backpropagation에서는 local derivative가 반복해서 곱해집니다.

Sigmoid derivative가 예를 들어 `0.2` 정도라면

```text
1 layer  : 0.2
2 layers : 0.04
3 layers : 0.008
...
```

처럼 앞쪽 layer로 갈수록 gradient가 빠르게 작아질 수 있습니다.

In [ ]:
depth = np.arange(1, 16)

for local_grad in [0.25, 0.2, 0.1]:
    grad_size = local_grad ** depth

    plt.plot(
        depth,
        grad_size,
        marker="o",
        label=f"local derivative={local_grad}"
    )

plt.xlabel("Number of Layers")
plt.ylabel("Gradient Magnitude")
plt.title("Repeated Multiplication of Small Derivatives")
plt.yscale("log")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 3. Sigmoid는 Zero-Centered가 아니다

Sigmoid output은 항상 양수입니다.

반면 tanh는 양수와 음수를 모두 출력하므로
평균이 0에 더 가까운 activation을 만들 수 있습니다.

작은 예제로 두 activation의 평균을 비교해 봅니다.

In [ ]:
rng = np.random.default_rng(0)

z = rng.normal(
    loc=0.0,
    scale=2.0,
    size=10000
)

sigmoid_h = sigmoid(z)
tanh_h = tanh(z)

print("Mean of sigmoid activation:", sigmoid_h.mean())
print("Mean of tanh activation   :", tanh_h.mean())

> ### ✅ 체크포인트
>
> Sigmoid activation은 항상 양수이므로 다음 layer의 weight gradient가 특정 방향으로 편향될 수 있습니다.  
> tanh는 zero-centered activation을 만들기 쉽다는 장점이 있습니다.

## 4. ReLU와 Dying ReLU

ReLU는

$$
\mathrm{ReLU}(x)=\max(0,x)
$$

이며 양수 영역에서는 derivative가 1이므로
sigmoid보다 gradient가 잘 전달되는 경우가 많습니다.

하지만 입력이 계속 음수이면 출력과 gradient가 모두 0이 될 수 있습니다.
이를 **dying ReLU**라고 합니다.

In [ ]:
z = np.array([-3.0, -1.0, -0.2, 0.0, 0.5, 2.0])

relu_output = relu(z)
relu_grad = (z > 0).astype(float)

print("z          :", z)
print("ReLU(z)    :", relu_output)
print("ReLU grad  :", relu_grad)

## 5. Leaky ReLU

Leaky ReLU는 음수 영역에서도 작은 slope를 유지합니다.

$$
f(x)=
\begin{cases}
x, & x \ge 0 \\
\alpha x, & x < 0
\end{cases}
$$

Lecture 9에서는 예시로 $\alpha=0.1$을 사용합니다.


In [ ]:
alpha = 0.1

leaky_output = leaky_relu(z, alpha=alpha)

leaky_grad = np.where(
    z >= 0,
    1.0,
    alpha
)

print("z              :", z)
print("LeakyReLU(z)   :", leaky_output)
print("LeakyReLU grad :", leaky_grad)

### 직접 해보기 1 — Leaky ReLU

다음을 바꾸어 보세요.

```python
alpha = 0.01
alpha = 0.1
alpha = 0.3
```

음수 영역의 activation과 gradient가 어떻게 달라지는지 확인하세요.

## 6. Activation Functions in Keras

Lecture 9의 Keras 예제를 그대로 사용해 activation layer를 구성해 봅니다.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense,
    Input,
    ReLU,
    LeakyReLU,
    ELU,
)

model = Sequential()

model.add(Input(shape=(12,)))

# ReLU
model.add(Dense(32))
model.add(ReLU())

# Leaky ReLU
model.add(Dense(16))
model.add(LeakyReLU(negative_slope=0.2))

# ELU
model.add(Dense(12))
model.add(ELU(alpha=1.0))

# Swish
model.add(Dense(8, activation="swish"))

# Linear output
model.add(Dense(1, activation="linear"))

model.summary()

### Lecture 9의 실전 선택 기준

```text
Hidden layers
→ ReLU: default
→ Leaky ReLU: deeper networks에서 대안

Output layers
→ Linear : regression
→ Sigmoid: binary classification
→ Softmax: multi-class classification
```

# Part II. Input Scaling

## 7. Normalization vs. Standardization

### Normalization

Feature의 범위를 제한합니다.

예:

```text
[0, 1]
[-1, 1]
```

### Standardization

Feature를 zero mean, unit variance로 만듭니다.

$$
x_{scaled}
=
\frac{x-\mu}{\sigma}
$$

Lecture 9의 핵심은 **feature별로 scaling한다**는 점입니다.

In [ ]:
from sklearn.preprocessing import (
    MinMaxScaler,
    StandardScaler,
)

X = np.array([
    [10.0,   1000.0],
    [20.0,   1500.0],
    [30.0,   2000.0],
    [40.0,   2500.0],
    [50.0,   3000.0],
])

minmax_scaler = MinMaxScaler()
standard_scaler = StandardScaler()

X_minmax = minmax_scaler.fit_transform(X)
X_standard = standard_scaler.fit_transform(X)

print("Raw data")
print(X)

print("\nMin-Max scaled")
print(X_minmax)

print("\nStandardized")
print(X_standard)

In [ ]:
print("Raw feature means:", X.mean(axis=0))
print("Raw feature stds :", X.std(axis=0))

print("\nStandardized means:", X_standard.mean(axis=0))
print("Standardized stds :", X_standard.std(axis=0))

### 확인할 내용

- Scaling은 **각 feature column별로** 수행됩니다.
- Standardization 후 각 feature의 평균은 약 0입니다.
- Standardization 후 각 feature의 표준편차는 약 1입니다.

## 8. Zero-Centering과 Scaling 시각화

두 feature의 scale이 매우 다른 synthetic data를 만들어 보겠습니다.

In [ ]:
rng = np.random.default_rng(1)

feature_1 = rng.normal(
    loc=20,
    scale=3,
    size=300
)

feature_2 = (
    100 * feature_1
    + rng.normal(0, 200, size=300)
)

X_raw = np.column_stack([
    feature_1,
    feature_2,
])

X_scaled = StandardScaler().fit_transform(
    X_raw
)

plt.scatter(
    X_raw[:, 0],
    X_raw[:, 1],
    s=10
)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Raw Data")
plt.grid(alpha=0.3)
plt.show()

plt.scatter(
    X_scaled[:, 0],
    X_scaled[:, 1],
    s=10
)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Standardized Data")
plt.grid(alpha=0.3)
plt.show()

### 확인할 내용

Scaling 후에는 각 feature의 numerical scale이 비슷해집니다.

이렇게 하면 특정 feature의 큰 numerical scale 때문에 optimization이 지나치게 영향을 받는 현상을 줄일 수 있습니다.

## 9. Train과 Test에서는 어떻게 Scaling해야 할까?

Lecture 9에서 가장 중요한 실전 규칙입니다.

```text
Training set → fit_transform()
Test set     → transform()
```

Training set에서만 mean/std 또는 min/max를 학습해야 합니다.

Test data까지 사용하여 scaler를 `fit`하면 **data leakage**가 발생합니다.

In [ ]:
X_train = np.array([
    [1.0, 100.0],
    [2.0, 120.0],
    [3.0, 140.0],
    [4.0, 160.0],
])

X_test = np.array([
    [5.0, 180.0],
    [6.0, 200.0],
])

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

X_test_scaled = scaler.transform(
    X_test
)

print("Training mean used by scaler:")
print(scaler.mean_)

print("\nScaled training data:")
print(X_train_scaled)

print("\nScaled test data:")
print(X_test_scaled)

## 10. MNIST Input Scaling

Lecture 9의 코드를 사용하여 MNIST image를 `[0, 1]` 범위로 scaling합니다.

In [ ]:
from sklearn.preprocessing import MinMaxScaler
# from sklearn.preprocessing import StandardScaler

from tensorflow.keras.datasets import mnist

# Load MNIST dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()

x_train = x_train.reshape(-1, 28 * 28)
x_test = x_test.reshape(-1, 28 * 28)

# Normalization: 0 ~ 1
scaler = MinMaxScaler()

# Standardization을 사용하려면:
# scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(
    x_train
)

x_test_scaled = scaler.transform(
    x_test
)

print(
    f"Min of normalized training data: "
    f"{x_train_scaled.min():.2f}"
)

print(
    f"Max of normalized training data: "
    f"{x_train_scaled.max():.2f}"
)

# Part III. Batch Normalization

## 11. Input Scaling vs. Activation Normalization

Input scaling은 **network에 들어가기 전 input feature**를 조정합니다.

Batch Normalization은 network 내부의 **pre-activation**을 normalization합니다.

```text
Input Scaling
x → network

Batch Normalization
... → z → BN → activation → ...
```

Lecture 9에서는 BatchNorm을 activation function **이전**에 적용하는 구조를 사용합니다.

## 12. Mini-Batch Matrix Representation

Mini-batch의 pre-activation matrix를

$$
\mathbf{Z}
\in
\mathbb{R}^{m \times p}
$$

라고 하겠습니다.

- $m$: batch size
- $p$: neuron 수
- 각 **row**: 하나의 sample
- 각 **column**: 하나의 neuron

Batch Normalization은 **각 column별로** mean과 variance를 계산합니다.

In [ ]:
Z = np.array([
    [ 1.0, 10.0, 100.0],
    [ 2.0, 12.0,  80.0],
    [ 3.0, 14.0, 120.0],
    [ 4.0, 16.0, 100.0],
])

print("Z shape:", Z.shape)
print(Z)

print("\nColumn means:")
print(Z.mean(axis=0))

print("\nColumn stds:")
print(Z.std(axis=0))

## 13. BatchNorm Step 1 — Neuron-wise Standardization

각 neuron(column)에 대해

$$
\mu_j
=
\frac{1}{m}
\sum_{i=1}^{m}
z_{ij}
$$

$$
\sigma_j^2
=
\frac{1}{m}
\sum_{i=1}^{m}
(z_{ij}-\mu_j)^2
$$

를 계산합니다.

그리고

$$
\hat{z}_{ij}
=
\frac{
z_{ij}-\mu_j
}{
\sqrt{\sigma_j^2+\epsilon}
}
$$

로 standardize합니다.

In [ ]:
eps = 1e-5

batch_mean = Z.mean(axis=0)
batch_var = Z.var(axis=0)

Z_hat = (
    Z - batch_mean
) / np.sqrt(
    batch_var + eps
)

print("Standardized Z:")
print(Z_hat)

print("\nColumn means after BN step 1:")
print(Z_hat.mean(axis=0))

print("\nColumn stds after BN step 1:")
print(Z_hat.std(axis=0))

### 확인할 내용

각 column의

```text
mean ≈ 0
std  ≈ 1
```

인지 확인하세요.

BatchNorm은 **batch 전체를 하나의 scalar mean/std로 정규화하는 것이 아니라, neuron별로 따로 계산**합니다.

## 14. BatchNorm Step 2 — Learnable Scaling and Shifting

BatchNorm은 standardization으로 끝나지 않습니다.

각 neuron에 learnable parameter $\gamma$, $\beta$를 적용합니다.

$$
s_{ij}
=
\gamma_j
\hat{z}_{ij}
+
\beta_j
$$

이렇게 하면 network가 필요할 경우
zero-mean/unit-variance에서 벗어난 적절한 scale과 shift를 다시 학습할 수 있습니다.

In [ ]:
gamma = np.array([1.0, 2.0, 0.5])
beta = np.array([0.0, 1.0, -1.0])

S = (
    gamma * Z_hat
    + beta
)

print("After scaling and shifting:")
print(S)

print("\nColumn means:")
print(S.mean(axis=0))

print("\nColumn stds:")
print(S.std(axis=0))

### 직접 해보기 2 — $\gamma$, $\beta$

다음을 바꾸어 보세요.

```python
gamma = [1, 1, 1]
beta  = [0, 0, 0]
```

또는

```python
gamma = [0.5, 2, 3]
beta  = [1, -1, 5]
```

표준화된 activation의 mean과 scale이 어떻게 변하는지 확인하세요.


## 15. Training Time과 Test Time

Training 중 BatchNorm은 현재 mini-batch의 mean/variance를 사용합니다.

하지만 test에서는 하나의 sample 또는 작은 batch가 들어올 수 있으므로
training 중 누적한 moving mean / moving variance를 사용합니다.

즉:

```text
Training
→ mini-batch statistics

Inference
→ moving statistics learned during training
```

## 16. Batch Normalization in Keras

Lecture 9의 MNIST 예제를 사용합니다.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense,
    BatchNormalization,
    Activation,
    Input,
)
from tensorflow.keras.datasets import mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()

x_train = (
    x_train
    .reshape(-1, 28 * 28)
    .astype("float32")
    / 255.0
)

x_test = (
    x_test
    .reshape(-1, 28 * 28)
    .astype("float32")
    / 255.0
)

model_bn = Sequential([
    Input(shape=(28 * 28,)),

    Dense(256),
    BatchNormalization(),
    Activation("relu"),

    Dense(128),
    BatchNormalization(),
    Activation("relu"),

    Dense(
        10,
        activation="softmax"
    ),
])

model_bn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model_bn.summary()

### Model Training

슬라이드의 설정은 `epochs=10`, `batch_size=32`입니다.

수업 중 시간을 줄이고 싶다면 `epochs=3` 정도로 먼저 실행해도 됩니다.

In [ ]:
# TODO:
# 전체 실습에서는 epochs=10,
# 빠른 확인은 epochs=3 정도로 실행해 보세요.

history_bn = model_bn.fit(
    x_train,
    y_train,
    epochs=3,
    batch_size=32,
    validation_data=(
        x_test,
        y_test
    ),
)

In [ ]:
plt.plot(
    history_bn.history["loss"],
    label="Train Loss"
)
plt.plot(
    history_bn.history["val_loss"],
    label="Validation Loss"
)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("MNIST with Batch Normalization")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 17. 직접 해보기 — BatchNorm을 제거하면?

같은 구조에서 BatchNormalization layer를 제거한 모델과 학습 속도를 비교할 수 있습니다.

아래 셀은 비교 실험용입니다.

In [ ]:
model_no_bn = Sequential([
    Input(shape=(28 * 28,)),
    Dense(256, activation="relu"),
    Dense(128, activation="relu"),
    Dense(10, activation="softmax"),
])

model_no_bn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history_no_bn = model_no_bn.fit(
    x_train,
    y_train,
    epochs=3,
    batch_size=32,
    validation_data=(
        x_test,
        y_test
    ),
)

In [ ]:
plt.plot(
    history_no_bn.history["val_loss"],
    label="Without BN"
)
plt.plot(
    history_bn.history["val_loss"],
    label="With BN"
)
plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.title("With vs. Without Batch Normalization")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 주의

작은 MNIST network에서는 BatchNorm이 항상 더 높은 최종 accuracy를 보장하지는 않습니다.

이번 비교의 목적은

- 학습 안정성
- convergence 속도
- initialization / learning rate에 대한 민감도

관점에서 BatchNorm의 역할을 이해하는 것입니다.

# Part IV. Layer Normalization

## 18. BatchNorm과 LayerNorm의 차이

BatchNorm과 LayerNorm은 모두 mean/variance를 이용해 activation을 standardize하지만
**어느 방향으로 통계량을 계산하는지가 다릅니다.**

Mini-batch activation matrix

$$
\mathbf{Z}
\in
\mathbb{R}^{m \times p}
$$

에서:

### Batch Normalization

각 neuron(column)별로 batch 방향에서 계산합니다.

```text
axis = 0
```

### Layer Normalization

각 sample(row)별로 feature/neuron 방향에서 계산합니다.

```text
axis = 1
```

In [ ]:
Z = np.array([
    [ 1.0, 10.0, 100.0],
    [ 2.0, 12.0,  80.0],
    [ 3.0, 14.0, 120.0],
    [ 4.0, 16.0, 100.0],
])

eps = 1e-5

# Batch Normalization style
bn_mean = Z.mean(
    axis=0,
    keepdims=True
)
bn_var = Z.var(
    axis=0,
    keepdims=True
)

Z_bn = (
    Z - bn_mean
) / np.sqrt(
    bn_var + eps
)

# Layer Normalization style
ln_mean = Z.mean(
    axis=1,
    keepdims=True
)
ln_var = Z.var(
    axis=1,
    keepdims=True
)

Z_ln = (
    Z - ln_mean
) / np.sqrt(
    ln_var + eps
)

print("Batch-normalized style:")
print(Z_bn)

print("\nLayer-normalized style:")
print(Z_ln)

In [ ]:
print("BN: column means")
print(Z_bn.mean(axis=0))

print("\nBN: column stds")
print(Z_bn.std(axis=0))

print("\nLN: row means")
print(Z_ln.mean(axis=1))

print("\nLN: row stds")
print(Z_ln.std(axis=1))

> ### ✅ 핵심 비교
>
> ```text
> BatchNorm
> → 같은 neuron을 여러 sample에서 비교
> → column-wise
>
> LayerNorm
> → 하나의 sample 안에서 여러 feature/neuron을 비교
> → row-wise
> ```

## 19. Layer Normalization in Keras

In [ ]:
from tensorflow.keras.layers import LayerNormalization

model_ln = Sequential([
    Input(shape=(28 * 28,)),

    Dense(256),
    LayerNormalization(),
    Activation("relu"),

    Dense(128),
    LayerNormalization(),
    Activation("relu"),

    Dense(
        10,
        activation="softmax"
    ),
])

model_ln.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model_ln.summary()

LayerNorm은 mini-batch 크기에 직접 의존하지 않기 때문에
sequence model과 Transformer 계열에서 매우 널리 사용됩니다.

이번 강의에서는 우선 **normalization axis의 차이**를 정확히 이해하는 것에 집중합니다.

# 20. 최종 실습

### Activation Functions

1. Sigmoid derivative가 가장 큰 입력 위치를 확인하세요.
2. `sigmoid`, `tanh` activation의 평균을 비교하세요.
3. ReLU의 음수 영역에서 gradient가 어떻게 되는지 확인하세요.
4. Leaky ReLU의 $\alpha$를 변경해 음수 영역의 gradient를 비교하세요.

### Input Scaling

5. `MinMaxScaler`와 `StandardScaler`를 같은 데이터에 적용해 보세요.
6. Standardization 후 각 feature의 mean/std를 확인하세요.
7. Test set에서 `fit_transform()`을 사용하면 왜 안 되는지 설명하세요.
8. MNIST에서 `MinMaxScaler` 대신 `StandardScaler`를 사용해 보세요.

### Batch Normalization

9. `Z` matrix의 각 column을 직접 standardize해 보세요.
10. $\gamma$, $\beta$를 바꾸고 출력의 mean/std 변화를 확인하세요.
11. BatchNorm이 activation function 이전에 위치하도록 model을 구성해 보세요.
12. BN이 있는 모델과 없는 모델의 learning curve를 비교하세요.

### Layer Normalization

13. BatchNorm과 LayerNorm의 normalization axis를 설명하세요.
14. `Z_bn.mean(axis=0)`와 `Z_ln.mean(axis=1)`이 각각 0에 가까운지 확인하세요.
15. Keras의 `LayerNormalization()`을 BatchNormalization과 교체해 보세요.

# 21. 정리

Lecture 9의 내용을 하나의 흐름으로 정리하면 다음과 같습니다.

```text
Input Scaling
      ↓
Linear / Affine Transformation
      ↓
Activation Normalization
      ↓
Activation Function
      ↓
Next Layer
```

### Activation Functions

| Function | 특징 |
|---|---|
| Sigmoid | bounded, but vanishing gradient / non-zero-centered |
| tanh | zero-centered, but saturation 가능 |
| ReLU | simple and effective, hidden layer의 기본 선택 |
| Leaky ReLU | negative region에서도 gradient 유지 |

### Input Scaling

```text
Normalization
→ fixed range

Standardization
→ zero mean, unit variance
```

가장 중요한 실전 규칙:

```text
Train → fit_transform()
Test  → transform()
```

### Batch Normalization

```text
Mini-batch matrix: (batch, neurons)

BatchNorm
→ neuron-wise
→ column-wise
→ batch statistics
```

Training에서는 mini-batch statistics를,
inference에서는 moving statistics를 사용합니다.

### Layer Normalization

```text
LayerNorm
→ sample-wise
→ feature/neuron direction
→ row-wise
```

### 꼭 기억할 것

1. **Activation function은 gradient flow에 직접 영향을 줍니다.**
2. **Input scaling은 optimization을 쉽게 만드는 기본 전처리입니다.**
3. **BatchNorm은 mini-batch 방향으로 neuron별 activation을 정규화합니다.**
4. **LayerNorm은 sample 내부 feature 방향으로 정규화합니다.**
5. **Normalization의 목적은 activation과 gradient의 scale을 더 안정적으로 유지하는 것입니다.**